# Lecture 1

## ChatGPT output with no guidance

In [10]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib


# Load dataset
df = pd.read_csv("/Users/dybl4375/Downloads/weatherHistory.csv")

# Parse date column into useful features
df["Formatted Date"] = pd.to_datetime(df["Formatted Date"], utc=True)

df["year"] = df["Formatted Date"].dt.year
df["month"] = df["Formatted Date"].dt.month
df["day"] = df["Formatted Date"].dt.day
df["hour"] = df["Formatted Date"].dt.hour

def run_RF(df):
    # Target column
    y = df["Summary"]
    
    # Drop target and columns that may leak text information
    X = df.drop(columns=["Summary", "Formatted Date", "Daily Summary"])
    
    # Identify feature types
    numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
    categorical_features = X.select_dtypes(include=["object"]).columns
    
    # Preprocessing
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])
    
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )
    
    # Random Forest model
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced_subsample",
            n_jobs=-1
        ))
    ])
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    
    # Train model
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Evaluate
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    # Save trained model
    joblib.dump(model, "weather_summary_random_forest.pkl")
    print("\nModel saved as weather_summary_random_forest.pkl")

run_RF(df)

ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

Let's help ChatGPT out a bit. The error comes from some categories that have too few data points. Before we got started, we should have checked what data catigories we had.

In [11]:
counts = df["Summary"].value_counts()
counts

Summary
Partly Cloudy                          31733
Mostly Cloudy                          28094
Overcast                               16597
Clear                                  10890
Foggy                                   7148
Breezy and Overcast                      528
Breezy and Mostly Cloudy                 516
Breezy and Partly Cloudy                 386
Dry and Partly Cloudy                     86
Windy and Partly Cloudy                   67
Light Rain                                63
Breezy                                    54
Windy and Overcast                        45
Humid and Mostly Cloudy                   40
Drizzle                                   39
Breezy and Foggy                          35
Windy and Mostly Cloudy                   35
Dry                                       34
Humid and Partly Cloudy                   17
Dry and Mostly Cloudy                     14
Rain                                      10
Windy                                      8
Hu

There are three catigories with just one entry. Let's remove those and see if the program runs.

In [13]:
df = df[df["Summary"].isin(counts[counts >= 2].index)]
run_RF(df)

Accuracy: 0.5933125972006221

Classification Report:
                          precision    recall  f1-score   support

                  Breezy       1.00      0.36      0.53        11
        Breezy and Foggy       1.00      0.71      0.83         7
Breezy and Mostly Cloudy       0.49      0.50      0.49       103
     Breezy and Overcast       0.64      0.75      0.69       106
Breezy and Partly Cloudy       0.58      0.65      0.61        77
                   Clear       0.60      0.31      0.41      2178
                 Drizzle       0.00      0.00      0.00         8
                     Dry       0.00      0.00      0.00         7
   Dry and Mostly Cloudy       0.00      0.00      0.00         3
   Dry and Partly Cloudy       0.60      0.18      0.27        17
                   Foggy       1.00      1.00      1.00      1430
 Humid and Mostly Cloudy       0.00      0.00      0.00         8
      Humid and Overcast       0.00      0.00      0.00         1
 Humid and Partly Clou

/opt/miniconda3/envs/dyl/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/miniconda3/envs/dyl/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/miniconda3/envs/dyl/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Model saved as weather_summary_random_forest.pkl


It worked! However, ChatGPT would not hold up to a peer review. If I was a reviewer on this model, I would ask the following questions.

1. Are all of these categories necessary and descriptive of our weather system?
2. Are the features the best variables needed to describe the weather?
3. Why were day, month, and year coded into the model directly?
4. Why were hyperparameters not tuned?
5. Why was there no iterative feature selection?
6. Why were data not normalized?